In [1]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy.stats import norm

import joblib
import pandas as pd


In [2]:
df = pd.read_csv("../csv/used_cars_1M_2025.csv")
df.head()

,id,country,city,brand,model,year,mileage_km,price_usd,fuel_type,transmission,horsepower,doors,color,condition_score,days_on_market,is_electric
0,1,USA,Adamport,Chevrolet,Silverado,2011.0,185945.0,5903.0,Hybrid,Manual,486.0,3.0,Silver,7.6,285.0,0.0
1,2,France,Mahe,Toyota,Corolla,2017.0,141520.0,9277.0,Plug-in Hybrid,Automatic,473.0,4.0,Brown,4.5,34.0,0.0
2,3,Germany,Hettstedt,BMW,5 Series,2016.0,139091.0,18918.0,Gasoline,Automatic,298.0,4.0,Blue,4.6,27.0,0.0
3,4,Germany,Kulmbach,Honda,CR-V,2007.0,258093.0,5058.0,Gasoline,Automatic,99.0,5.0,White,4.6,362.0,0.0
4,5,USA,Port Cory,Hyundai,Elantra,2017.0,147560.0,16954.0,Hybrid,Automatic,236.0,5.0,Yellow,8.5,359.0,1.0


In [3]:
df.drop("id", axis=1, inplace=True)

In [4]:
print(df.columns)

Index(['country', 'city', 'brand', 'model', 'year', 'mileage_km', 'price_usd',
       'fuel_type', 'transmission', 'horsepower', 'doors', 'color',
       'condition_score', 'days_on_market', 'is_electric'],
      dtype='object')


In [10]:
df = df.dropna(subset=["price_usd"])
X = df.drop("price_usd", axis=1)
y = df["price_usd"]

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include="object").columns.tolist()

print("Colonnes numériques:", num_cols)
print("Colonnes catégorielles:", cat_cols)

num_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_preprocess, num_cols),
        ("cat", cat_preprocess, cat_cols)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

lr = LinearRegression()

pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", lr)
], verbose=True)

print(pipeline)


Colonnes numériques: ['year', 'mileage_km', 'horsepower', 'doors', 'condition_score', 'days_on_market', 'is_electric']
Colonnes catégorielles: ['country', 'city', 'brand', 'model', 'fuel_type', 'transmission', 'color']
Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['year', 'mileage_km',
                                                   'horsepower', 'doors',
                                                   'condition_score',
                                                   'days_on_market',
                                                   'is_electri

In [13]:
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("R2:", r2)


[Pipeline] ........ (step 1 of 2) Processing preprocess, total=   2.0s
[Pipeline] ............. (step 2 of 2) Processing model, total=   0.2s
RMSE: 21886.896283989863
R2: 0.17141974112350866


In [14]:
feature_names = pipeline.named_steps["preprocess"].get_feature_names_out()
lr_model = pipeline.named_steps["model"]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": lr_model.coef_
}).sort_values(by="coefficient", key=abs, ascending=False)

print(coef_df.to_string(index=False))


        feature  coefficient
           year 21073.669931
     mileage_km 12672.608107
   transmission   -74.504546
 days_on_market    44.122259
      fuel_type   -41.534803
          doors   -30.938050
    is_electric   -15.554437
     horsepower    12.941375
condition_score     6.979522
          brand    -6.491171
        country     2.793465
          color    -2.699291
          model     2.459156
           city    -0.000247


In [15]:
joblib.dump(pipeline, "linear_regression_cars.pkl")


['linear_regression_cars.pkl']